# DenseOnly vs DenseGraph + Generate Evaluation

Notebook này chạy 500 câu hỏi benchmark, so sánh 2 mode:

- `DenseOnly`: FAISS dense retrieval rồi generate.
- `DenseGraph`: FAISS dense seed, GraphExpansion mở rộng context, rồi generate.

Output có cả top-k chunks và generated answer cho từng câu hỏi.

In [ ]:
!pip -q install faiss-cpu sentence-transformers pandas psutil openai

In [ ]:
from pathlib import Path
import contextlib
import io
import json
import math
import os
import re
import shutil
import sys
import time

import pandas as pd
import psutil
import torch
from kaggle_secrets import UserSecretsClient
from openai import OpenAI
from sentence_transformers import SentenceTransformer

In [ ]:
def print_ram(label):
    mem = psutil.virtual_memory()
    process_ram = psutil.Process(os.getpid()).memory_info().rss / 1024**3
    print(f'[{label}] process={process_ram:.2f}GB | free={mem.available / 1024**3:.2f}GB')

print_ram('Start')

In [ ]:
REPO_DIR = Path('/kaggle/working/TextMining')

if not REPO_DIR.exists():
    !git clone -q https://github.com/PhuongThao-2005/TextMining.git /kaggle/working/TextMining

sys.path.insert(0, str(REPO_DIR / 'src'))

from knowledge_graph import GraphExpansion, load_knowledge_graph
from retrieval.sqlite_faiss_store import SQLitePayloadFaissVectorStore

print('Repo ready:', REPO_DIR)

In [ ]:
KAGGLE_INPUT = Path('/kaggle/input')

FAISS_INPUT_DIR = KAGGLE_INPUT / 'datasets/kittrntunk/faiss-chunk-meta'
FAISS_WORK_DIR = Path('/kaggle/working/faiss-chunk-meta')
GRAPH_PATH = KAGGLE_INPUT / 'datasets/nguyenlethienlyy/legalrag-knowledge-graph/knowledge_graph.gpickle'
QA_PATH = KAGGLE_INPUT / 'datasets/nguyenlethienlyy/legalrag-text-mining-benchmark/qa_final.jsonl'

OUT_DIR = Path('/kaggle/working/evaluation_runs/ablation/graph_dense_generate_eval')

EMBEDDING_MODEL = 'intfloat/multilingual-e5-large'
GENERATION_MODEL = 'gpt-4o-mini'
BASE_URL = 'https://api.shopaikey.com/v1'

SEARCH_K = 50
FINAL_TOP_N = 10
GRAPH_SEED_N = 5
GRAPH_MAX_HOP = 1
GRAPH_CONTEXT = 10
FILTER_PROFILE = 'broad'
SAMPLE_LIMIT = None  # đổi thành 10 để smoke test trước; None là chạy đủ 500 câu

for path in [FAISS_INPUT_DIR / 'index.faiss', FAISS_INPUT_DIR / 'payloads.jsonl', FAISS_INPUT_DIR / 'id_map.json', FAISS_INPUT_DIR / 'payload_cache.sqlite', GRAPH_PATH, QA_PATH]:
    if not path.exists():
        raise FileNotFoundError(path)

OUT_DIR.mkdir(parents=True, exist_ok=True)

print('FAISS_INPUT_DIR =', FAISS_INPUT_DIR)
print('GRAPH_PATH      =', GRAPH_PATH)
print('QA_PATH         =', QA_PATH)
print('OUT_DIR         =', OUT_DIR)

In [ ]:
FAISS_WORK_DIR.mkdir(parents=True, exist_ok=True)

for name in ['index.faiss', 'payloads.jsonl', 'id_map.json']:
    src = FAISS_INPUT_DIR / name
    dst = FAISS_WORK_DIR / name
    if not dst.exists():
        os.symlink(src, dst)

cache_src = FAISS_INPUT_DIR / 'payload_cache.sqlite'
cache_dst = FAISS_WORK_DIR / 'payload_cache.sqlite'
if not cache_dst.exists():
    shutil.copy2(cache_src, cache_dst)

print('FAISS working folder ready:', FAISS_WORK_DIR)
print_ram('After preparing FAISS folder')

In [ ]:
qa_rows = []
with QA_PATH.open('r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            qa_rows.append(json.loads(line))

if SAMPLE_LIMIT is not None:
    qa_rows = qa_rows[:SAMPLE_LIMIT]

answerable_count = sum(bool((row.get('ground_truth') or {}).get('chunk_ids')) for row in qa_rows)

print('QA total      =', len(qa_rows))
print('answerable    =', answerable_count)
print('unanswerable  =', len(qa_rows) - answerable_count)

In [ ]:
t0 = time.perf_counter()
store = SQLitePayloadFaissVectorStore.load(FAISS_WORK_DIR)
print(f'Loaded FAISS store: {store.total_vectors:,} vectors, {time.perf_counter() - t0:.1f}s')
print_ram('After FAISS store')

In [ ]:
t0 = time.perf_counter()
graph = load_knowledge_graph(GRAPH_PATH).graph
graph_expansion = GraphExpansion(graph)
print(f'Loaded graph: {len(graph.chunks):,} chunks, {time.perf_counter() - t0:.1f}s')
print_ram('After graph')

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

t0 = time.perf_counter()
embedder = SentenceTransformer(EMBEDDING_MODEL, device=device)
print(f'Embedder ready: {EMBEDDING_MODEL} | device={device} | {time.perf_counter() - t0:.1f}s')
print_ram('After embedder')

In [ ]:
api_key = UserSecretsClient().get_secret('OPENAI_API_KEY').strip()
client = OpenAI(api_key=api_key, base_url=BASE_URL)

print('Generator ready:', GENERATION_MODEL)
print('BASE_URL:', BASE_URL)

In [ ]:
def filters_for(profile):
    if profile == 'current_law':
        return {'validity_group': {'in': ['active', 'partial', 'future']}}
    if profile == 'historical':
        return {'validity_group': {'in': ['expired', 'active', 'partial']}}
    return {'validity_group': {'in': ['active', 'partial', 'future', 'expired', 'unknown']}}

def row_from_payload(payload, rank, score=None):
    return {
        'rank': rank,
        'score': None if score is None else float(score),
        'chunk_id': str(payload.get('chunk_id') or ''),
        'document_id': str(payload.get('id_str') or ''),
        'provision_id': str(payload.get('parent_unit_id') or ''),
        'citation': payload.get('citation_anchor') or payload.get('citation_label') or '',
        'title': payload.get('title') or '',
        'validity': payload.get('validity_group') or '',
        'text': payload.get('chunk_text') or '',
    }

def retrieve_dense(question):
    t0 = time.perf_counter()
    vector = embedder.encode(['query: ' + question], normalize_embeddings=True).astype('float32')[0].tolist()
    embed_latency = time.perf_counter() - t0

    t0 = time.perf_counter()
    with contextlib.redirect_stdout(io.StringIO()):
        hits = store.search(vector, limit=SEARCH_K, score_threshold=0.0, filters=filters_for(FILTER_PROFILE))
    search_latency = time.perf_counter() - t0

    rows = []
    seen = set()
    for hit in hits:
        chunk_id = str(hit.payload.get('chunk_id') or hit.point_id)
        if chunk_id in seen:
            continue
        seen.add(chunk_id)
        rows.append(row_from_payload(hit.payload, len(rows) + 1, hit.score))
        if len(rows) >= FINAL_TOP_N:
            break

    return rows, embed_latency, search_latency

In [ ]:
def expand_with_graph(seed_rows):
    t0 = time.perf_counter()
    seed_ids = [row['chunk_id'] for row in seed_rows]
    expanded = graph_expansion.expand(seed_ids, max_hop=GRAPH_MAX_HOP, max_context=GRAPH_CONTEXT)
    chunk_ids = list(expanded.ordered_context_chunks)

    if not chunk_ids:
        return seed_rows, time.perf_counter() - t0, 0, list(expanded.warnings)

    with contextlib.redirect_stdout(io.StringIO()):
        hits = store.scroll({'chunk_id': {'in': chunk_ids}}, limit=len(chunk_ids))

    payload_by_chunk_id = {str(hit.payload.get('chunk_id')): hit.payload for hit in hits}
    rows = []
    for chunk_id in chunk_ids:
        payload = payload_by_chunk_id.get(chunk_id)
        if payload:
            rows.append(row_from_payload(payload, len(rows) + 1))

    seed_set = set(seed_ids)
    graph_added = len([row for row in rows if row['chunk_id'] not in seed_set])
    return rows or seed_rows, time.perf_counter() - t0, graph_added, list(expanded.warnings)

In [ ]:
def format_context(rows):
    blocks = []
    for row in rows:
        blocks.append(
            f"[{row['rank']}] {row['citation']} - {row['title']}\n"
            f"chunk_id={row['chunk_id']}\n"
            f"{row['text']}"
        )
    return '\n\n'.join(blocks)

def generate_answer(question, context_rows):
    prompt = f"""Bạn là trợ lý pháp lý tiếng Việt trong hệ thống RAG.

Hãy trả lời QUESTION dựa trên CONTEXT được cung cấp.

Nguyên tắc:
- Chỉ dùng thông tin có trong CONTEXT, không dùng kiến thức bên ngoài.
- Không bịa thêm căn cứ, điều kiện, ngoại lệ hoặc số điều nếu CONTEXT không nêu.
- Nếu CONTEXT không đủ thông tin để trả lời, hãy nói: "Không có đủ thông tin trong ngữ cảnh được cung cấp."
- Nếu CONTEXT chỉ trả lời được một phần câu hỏi, hãy nói rõ phạm vi đó.

Cách trả lời:
- Trả lời tự nhiên, rõ ràng, bằng tiếng Việt có dấu.
- Ưu tiên trả lời trực tiếp trước, giải thích ngắn sau nếu cần.
- Khi có căn cứ pháp lý trong CONTEXT, hãy nêu căn cứ ở cuối câu trả lời.
- Không trình bày quá trình suy luận nội bộ.

QUESTION:
{question}

CONTEXT:
{format_context(context_rows)}

Trả lời:""".strip()

    t0 = time.perf_counter()
    response = client.chat.completions.create(
        model=GENERATION_MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0,
    )
    latency = time.perf_counter() - t0
    return response.choices[0].message.content.strip(), latency

In [ ]:
def normalize_text(text):
    text = (text or '').lower()
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def token_f1(prediction, reference):
    pred_tokens = normalize_text(prediction).split()
    ref_tokens = normalize_text(reference).split()
    if not pred_tokens or not ref_tokens:
        return float(pred_tokens == ref_tokens)
    common = {}
    for token in pred_tokens:
        common[token] = min(pred_tokens.count(token), ref_tokens.count(token))
    overlap = sum(common.values())
    if overlap == 0:
        return 0.0
    precision = overlap / len(pred_tokens)
    recall = overlap / len(ref_tokens)
    return 2 * precision * recall / (precision + recall)

def rouge_l(prediction, reference):
    a = normalize_text(prediction).split()
    b = normalize_text(reference).split()
    if not a or not b:
        return float(a == b)
    dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            dp[i][j] = dp[i-1][j-1] + 1 if a[i-1] == b[j-1] else max(dp[i-1][j], dp[i][j-1])
    lcs = dp[-1][-1]
    precision = lcs / len(a)
    recall = lcs / len(b)
    return 2 * precision * recall / (precision + recall) if precision + recall else 0.0

def exact_match(prediction, reference):
    return float(normalize_text(prediction) == normalize_text(reference))

def has_fallback_answer(answer):
    return 'không có đủ thông tin' in normalize_text(answer)

In [ ]:
def recall_at_k(retrieved, relevant, k):
    relevant = set(relevant)
    return len(set(retrieved[:k]) & relevant) / len(relevant) if relevant else 0.0

def hit_at_k(retrieved, relevant, k):
    return 1.0 if set(retrieved[:k]) & set(relevant) else 0.0

def build_output_row(qa, mode, context_rows, answer, timings, graph_info):
    gt = qa.get('ground_truth') or {}
    relevant = gt.get('chunk_ids') or []
    retrieved = [row['chunk_id'] for row in context_rows]
    reference = qa.get('reference_answer') or ''
    answerable = bool(relevant)

    return {
        'qa_id': qa.get('qa_id'),
        'mode': mode,
        'question': qa.get('question'),
        'reference_answer': reference,
        'generated_answer': answer,
        'answer_type': qa.get('answer_type'),
        'category': qa.get('category'),
        'difficulty': qa.get('difficulty'),
        'is_answerable': answerable,
        'ground_truth_chunk_ids': relevant,
        'top_k_chunks': context_rows,
        'retrieved_chunk_ids': retrieved,
        'retrieved_count': len(retrieved),
        'hit@10': hit_at_k(retrieved, relevant, 10),
        'recall@10': recall_at_k(retrieved, relevant, 10),
        'exact_match': exact_match(answer, reference),
        'token_f1': token_f1(answer, reference),
        'rouge_l': rouge_l(answer, reference),
        'unanswerable_correct': (not answerable and has_fallback_answer(answer)),
        **timings,
        **graph_info,
    }

In [ ]:
all_cases = []
run_start = time.perf_counter()

for index, qa in enumerate(qa_rows, start=1):
    question = qa.get('question') or ''
    dense_rows, embed_latency, search_latency = retrieve_dense(question)

    dense_answer, dense_gen_latency = generate_answer(question, dense_rows)
    all_cases.append(build_output_row(
        qa,
        'DenseOnly',
        dense_rows,
        dense_answer,
        {
            'embedding_latency_sec': embed_latency,
            'dense_search_latency_sec': search_latency,
            'graph_latency_sec': 0.0,
            'generation_latency_sec': dense_gen_latency,
            'total_latency_sec': embed_latency + search_latency + dense_gen_latency,
        },
        {
            'seed_count': 0,
            'graph_added_count': 0,
            'graph_warning_count': 0,
            'graph_warnings': [],
        },
    ))

    graph_rows, graph_latency, graph_added_count, warnings = expand_with_graph(dense_rows[:GRAPH_SEED_N])
    graph_answer, graph_gen_latency = generate_answer(question, graph_rows)
    all_cases.append(build_output_row(
        qa,
        'DenseGraph',
        graph_rows,
        graph_answer,
        {
            'embedding_latency_sec': embed_latency,
            'dense_search_latency_sec': search_latency,
            'graph_latency_sec': graph_latency,
            'generation_latency_sec': graph_gen_latency,
            'total_latency_sec': embed_latency + search_latency + graph_latency + graph_gen_latency,
        },
        {
            'seed_count': min(len(dense_rows), GRAPH_SEED_N),
            'graph_added_count': graph_added_count,
            'graph_warning_count': len(warnings),
            'graph_warnings': warnings,
        },
    ))

    if index % 10 == 0 or index == len(qa_rows):
        elapsed = time.perf_counter() - run_start
        print(f'{index}/{len(qa_rows)} done | elapsed={elapsed/60:.1f} min')
        print_ram(f'After {index} questions')

total_benchmark_time = time.perf_counter() - run_start
print(f'Done in {total_benchmark_time/60:.1f} minutes')

In [ ]:
def average(rows, key):
    values = [row.get(key) for row in rows if isinstance(row.get(key), (int, float))]
    return sum(values) / len(values) if values else 0.0

def summarize(cases, label, answerable_only=False):
    rows = [row for row in cases if row['is_answerable']] if answerable_only else list(cases)
    summary = []
    for mode in ['DenseOnly', 'DenseGraph']:
        mode_rows = [row for row in rows if row['mode'] == mode]
        unanswerable_rows = [row for row in mode_rows if not row['is_answerable']]
        summary.append({
            'summary_set': label,
            'mode': mode,
            'evaluated': len(mode_rows),
            'answerable': sum(row['is_answerable'] for row in mode_rows),
            'unanswerable': len(unanswerable_rows),
            'hit@10': average(mode_rows, 'hit@10'),
            'recall@10': average(mode_rows, 'recall@10'),
            'exact_match': average(mode_rows, 'exact_match'),
            'token_f1': average(mode_rows, 'token_f1'),
            'rouge_l': average(mode_rows, 'rouge_l'),
            'unanswerable_accuracy': average(unanswerable_rows, 'unanswerable_correct') if unanswerable_rows else None,
            'avg_retrieved_count': average(mode_rows, 'retrieved_count'),
            'avg_graph_added_count': average(mode_rows, 'graph_added_count'),
            'avg_embedding_latency_sec': average(mode_rows, 'embedding_latency_sec'),
            'avg_dense_search_latency_sec': average(mode_rows, 'dense_search_latency_sec'),
            'avg_graph_latency_sec': average(mode_rows, 'graph_latency_sec'),
            'avg_generation_latency_sec': average(mode_rows, 'generation_latency_sec'),
            'avg_total_latency_sec': average(mode_rows, 'total_latency_sec'),
            'median_total_latency_sec': pd.Series([row['total_latency_sec'] for row in mode_rows]).median() if mode_rows else 0.0,
        })
    return pd.DataFrame(summary)

summary_all = summarize(all_cases, 'all_cases', answerable_only=False)
summary_answerable = summarize(all_cases, 'answerable_only', answerable_only=True)

display(summary_all)
display(summary_answerable)

In [ ]:
def write_jsonl(path, rows):
    with path.open('w', encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')

def csv_safe(df):
    out = df.copy()
    for col in out.columns:
        if out[col].map(lambda x: isinstance(x, (list, dict))).any():
            out[col] = out[col].map(lambda x: json.dumps(x, ensure_ascii=False) if isinstance(x, (list, dict)) else x)
    return out

cases_df = pd.DataFrame(all_cases)

write_jsonl(OUT_DIR / 'e2e_cases.jsonl', all_cases)
csv_safe(cases_df).to_csv(OUT_DIR / 'e2e_cases.csv', index=False, encoding='utf-8-sig')
summary_all.to_csv(OUT_DIR / 'summary_all_cases.csv', index=False, encoding='utf-8-sig')
summary_answerable.to_csv(OUT_DIR / 'summary_answerable_only.csv', index=False, encoding='utf-8-sig')

manifest = {
    'run_name': 'graph_dense_generate_eval',
    'modes': ['DenseOnly', 'DenseGraph'],
    'benchmark_path': str(QA_PATH),
    'faiss_dir': str(FAISS_INPUT_DIR),
    'graph_path': str(GRAPH_PATH),
    'embedding_model': EMBEDDING_MODEL,
    'generation_model': GENERATION_MODEL,
    'base_url': BASE_URL,
    'search_k': SEARCH_K,
    'final_top_n': FINAL_TOP_N,
    'graph_seed_n': GRAPH_SEED_N,
    'graph_max_hop': GRAPH_MAX_HOP,
    'graph_context': GRAPH_CONTEXT,
    'filter_profile': FILTER_PROFILE,
    'sample_limit': SAMPLE_LIMIT,
    'qa_total': len(qa_rows),
    'answerable_total': answerable_count,
    'unanswerable_total': len(qa_rows) - answerable_count,
    'total_benchmark_time_sec': total_benchmark_time,
}

with (OUT_DIR / 'manifest.json').open('w', encoding='utf-8') as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print('Saved outputs to:', OUT_DIR)
for path in sorted(OUT_DIR.iterdir()):
    print(' -', path.name)

In [ ]:
ZIP_PATH = Path('/kaggle/working/graph_dense_eval_outputs.zip')

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

shutil.make_archive(str(ZIP_PATH.with_suffix('')), 'zip', OUT_DIR)

print('Created:', ZIP_PATH)
print(f'Size: {ZIP_PATH.stat().st_size / 1024**2:.1f} MB')